# TBD Phase 2 26L: Performance & Computing Models

This notebook starts the 26L edition of Phase 2. The previous notebook is intentionally kept unchanged as `tbd_phase_2.ipynb`.

This file currently contains only **Task 1: Reproducible Dataset & Data Layout**. Later tasks will use this dataset to compare Pandas, Polars, DuckDB, and PySpark.

## Why Task 1 changed

The previous edition generated one synthetic Parquet file with random social-media-like data. That was enough for a first benchmark, but it hid several problems that matter in real Big Data work:

- benchmark results are not reproducible without a fixed seed and recorded environment,
- data layout can dominate query performance,
- skew, nulls, late events, and schema changes are common in production datasets,
- large text columns can make data generation slow without improving the analytical benchmark.

In this task you will create a reproducible dataset with two Parquet layouts and small dimension tables for later joins.

## Polars API note for 26L

The new edition should use the modern Polars execution-engine API in later tasks:

```python
lazy_frame.collect(engine="streaming")
```

Do not use the older `collect(streaming=True)` style in new code. This Task 1 notebook records the installed Polars version in the dataset manifest so that later reports can compare the course environment against the previous semester.

## Prerequisites

Run this once in your notebook environment. If the course image already provides the packages, the command should finish quickly.

In [ ]:
%pip install -U "polars>=1.30" "numpy>=2.0" "pyarrow>=16" "psutil>=5.9"

## Task 1: Reproducible Dataset & Data Layout

### Goal

Generate a realistic analytical dataset that can be reused by all later Phase 2 tasks.

Your dataset must include:

- deterministic generation from a seed,
- event data with skewed users, null values, late-arriving events, and a simple schema evolution marker,
- two physical layouts: unpartitioned Parquet and date-partitioned Parquet,
- two small dimension tables for later joins,
- a JSON manifest describing versions, row counts, file counts, and dataset size.

### Student deliverables

Submit a short report with:

1. selected dataset scale and row count,
2. installed versions of Python, Polars, NumPy, and PyArrow,
3. number of Parquet files and total size for each layout,
4. a short explanation of why partitioning by `event_date` should help some queries and hurt others,
5. one paragraph describing how skew, nulls, and late events may affect benchmark results in later tasks.

Do not commit generated data to Git.

In [ ]:
from __future__ import annotations

import json
import os
import platform
import shutil
import sys
import time
from datetime import date, datetime, timezone
from pathlib import Path

import numpy as np
import polars as pl
import pyarrow as pa
import psutil

print("Python:", sys.version.split()[0])
print("Polars:", pl.__version__)
print("NumPy:", np.__version__)
print("PyArrow:", pa.__version__)
print("CPU cores:", psutil.cpu_count(logical=True))
print("RAM GiB:", round(psutil.virtual_memory().total / 2**30, 2))

## Configuration

Start with `debug` or `small`. Use `medium` only after your notebook environment can generate and read the smaller dataset reliably. `large` is intended for stronger local machines or cluster-backed environments.

In [ ]:
SEED = 20260425

SCALE = "small"  # allowed: debug, small, medium, large
SCALE_ROWS = {
    "debug": 100_000,
    "small": 1_000_000,
    "medium": 10_000_000,
    "large": 50_000_000,
}

NUM_ROWS = SCALE_ROWS[SCALE]
CHUNK_SIZE = 500_000
NUM_USERS = 250_000
NUM_CAMPAIGNS = 500

START_DATE = np.datetime64("2025-01-01T00:00:00", "s")
END_DATE = np.datetime64("2025-04-01T00:00:00", "s")
COMPRESSION = "zstd"

BASE_DIR = Path("../data/phase2_26L").resolve()
UNPARTITIONED_DIR = BASE_DIR / "events_unpartitioned"
PARTITIONED_DIR = BASE_DIR / "events_partitioned_by_date"
DIM_DIR = BASE_DIR / "dimensions"
MANIFEST_PATH = BASE_DIR / "dataset_manifest.json"

OVERWRITE = False

CATEGORY_VALUES = np.array(["tech", "health", "travel", "food", "fashion", "politics", "sports"])
COUNTRY_VALUES = np.array(["PL", "DE", "FR", "UK", "US", "BR", "JP", "IN"])
DEVICE_VALUES = np.array(["mobile", "desktop", "tablet", "bot"])
LANGUAGE_VALUES = np.array(["pl", "en", "de", "fr", "es", "pt", "ja"])
TAG_VALUES = np.array([
    "ai", "spark", "polars", "duckdb", "cloud", "mlops", "security", "etl",
    "streaming", "sql", "python", "data", "infra", "analytics", "benchmark"
])

print(f"Generating scale={SCALE!r}, rows={NUM_ROWS:,}")
print("Output directory:", BASE_DIR)

## Dimension tables

The event table will reference users and campaigns. These tables are intentionally small enough to be used in broadcast joins in Spark and normal joins in Polars/DuckDB.

In [ ]:
def build_users(num_users: int, seed: int) -> pl.DataFrame:
    rng = np.random.default_rng(seed)
    signup_offsets = rng.integers(0, 365 * 4, size=num_users, dtype=np.int32)
    signup_date = np.datetime64("2021-01-01", "D") + signup_offsets.astype("timedelta64[D]")

    return pl.DataFrame(
        {
            "user_id": np.arange(1, num_users + 1, dtype=np.int32),
            "home_country": rng.choice(COUNTRY_VALUES, size=num_users, p=[0.20, 0.14, 0.12, 0.10, 0.20, 0.08, 0.06, 0.10]),
            "user_segment": rng.choice(["new", "regular", "creator", "power"], size=num_users, p=[0.25, 0.55, 0.15, 0.05]),
            "signup_date": signup_date,
            "is_verified": rng.random(num_users) < 0.08,
        }
    )


def build_campaigns(num_campaigns: int, seed: int) -> pl.DataFrame:
    rng = np.random.default_rng(seed)
    campaign_start = np.datetime64("2024-12-15", "D") + rng.integers(0, 90, size=num_campaigns).astype("timedelta64[D]")

    return pl.DataFrame(
        {
            "campaign_id": np.arange(1, num_campaigns + 1, dtype=np.int32),
            "campaign_channel": rng.choice(["organic", "search", "social", "display", "email"], size=num_campaigns),
            "campaign_budget_eur": rng.lognormal(mean=8.0, sigma=0.8, size=num_campaigns).round(2),
            "campaign_start_date": campaign_start,
        }
    )


users_df = build_users(NUM_USERS, SEED + 1)
campaigns_df = build_campaigns(NUM_CAMPAIGNS, SEED + 2)

users_df.head(), campaigns_df.head()

## Event generator

The event generator uses chunks. This keeps memory usage predictable and makes the same code usable for larger scales.

Important properties:

- **skewed users**: a small group of users produces a large share of events,
- **late events**: `ingest_ts` can be hours or days after `event_ts`,
- **nulls**: selected categorical fields contain missing values,
- **schema marker**: `schema_version` and `experiment_group` simulate a simple schema evolution,
- **nested-ish field**: `tags` is a list-like column useful for later `explode` tasks.

In [ ]:
def skewed_user_ids(rng: np.random.Generator, n: int, num_users: int) -> np.ndarray:
    hot_user_count = max(100, int(num_users * 0.01))
    hot_mask = rng.random(n) < 0.50
    user_ids = rng.integers(hot_user_count + 1, num_users + 1, size=n, dtype=np.int32)
    user_ids[hot_mask] = rng.integers(1, hot_user_count + 1, size=int(hot_mask.sum()), dtype=np.int32)
    return user_ids


def random_tags(rng: np.random.Generator, n: int) -> list[list[str]]:
    tag_counts = rng.integers(1, 4, size=n)
    tag_ids = rng.integers(0, len(TAG_VALUES), size=(n, 3))
    return [[str(TAG_VALUES[tag_ids[i, j]]) for j in range(tag_counts[i])] for i in range(n)]


def generate_event_chunk(start_id: int, n: int, rng: np.random.Generator) -> pl.DataFrame:
    total_seconds = int((END_DATE - START_DATE) / np.timedelta64(1, "s"))
    event_offsets = rng.integers(0, total_seconds, size=n, dtype=np.int64)
    event_ts = START_DATE + event_offsets.astype("timedelta64[s]")

    normal_lag = rng.exponential(scale=180, size=n).astype(np.int64)
    late_mask = rng.random(n) < 0.03
    normal_lag[late_mask] += rng.integers(6 * 3600, 7 * 24 * 3600, size=int(late_mask.sum()), dtype=np.int64)
    ingest_ts = event_ts + normal_lag.astype("timedelta64[s]")

    views = rng.lognormal(mean=6.5, sigma=1.2, size=n).astype(np.int64)
    likes = rng.binomial(np.minimum(views, 20_000), p=0.035).astype(np.int32)
    shares = rng.binomial(np.minimum(views, 20_000), p=0.006).astype(np.int32)

    has_campaign = rng.random(n) < 0.65
    campaign_id = rng.integers(1, NUM_CAMPAIGNS + 1, size=n, dtype=np.int32)

    country = rng.choice(COUNTRY_VALUES, size=n, p=[0.22, 0.13, 0.11, 0.10, 0.19, 0.07, 0.07, 0.11]).astype(object)
    country[rng.random(n) < 0.015] = None

    device = rng.choice(DEVICE_VALUES, size=n, p=[0.68, 0.22, 0.08, 0.02]).astype(object)
    device[rng.random(n) < 0.01] = None

    schema_version = np.where(event_offsets >= int(total_seconds * 0.60), 2, 1).astype(np.int8)
    experiment_group = rng.choice(["control", "variant_a", "variant_b"], size=n, p=[0.50, 0.25, 0.25]).astype(object)
    experiment_group[schema_version == 1] = None

    df = pl.DataFrame(
        {
            "post_id": np.arange(start_id, start_id + n, dtype=np.int64),
            "user_id": skewed_user_ids(rng, n, NUM_USERS),
            "event_ts": event_ts,
            "ingest_ts": ingest_ts,
            "category": rng.choice(CATEGORY_VALUES, size=n, p=[0.18, 0.12, 0.12, 0.11, 0.10, 0.17, 0.20]),
            "country": country,
            "device": device,
            "language": rng.choice(LANGUAGE_VALUES, size=n, p=[0.22, 0.36, 0.10, 0.08, 0.08, 0.08, 0.08]),
            "campaign_id_raw": campaign_id,
            "has_campaign": has_campaign,
            "views": views,
            "likes": likes,
            "shares": shares,
            "latency_ms": rng.lognormal(mean=4.9, sigma=0.55, size=n).round(3),
            "upload_error": rng.random(n) < 0.025,
            "content_length": rng.gamma(shape=2.2, scale=60.0, size=n).astype(np.int32),
            "tags": random_tags(rng, n),
            "schema_version": schema_version,
            "experiment_group": experiment_group,
        }
    )

    return df.with_columns(
        pl.col("event_ts").dt.date().alias("event_date"),
        pl.when(pl.col("has_campaign"))
        .then(pl.col("campaign_id_raw"))
        .otherwise(None)
        .cast(pl.Int32)
        .alias("campaign_id"),
        (pl.col("ingest_ts") - pl.col("event_ts")).dt.total_seconds().cast(pl.Int64).alias("ingest_lag_seconds"),
    ).drop(["campaign_id_raw", "has_campaign"])

## Write the dataset

This step creates:

- `events_unpartitioned/*.parquet` - simple layout with chunk files,
- `events_partitioned_by_date/event_date=YYYY-MM-DD/*.parquet` - Hive-style date partitioning,
- `dimensions/users.parquet`,
- `dimensions/campaigns.parquet`,
- `dataset_manifest.json`.

The partitioned layout intentionally produces more files. Later tasks will measure when that helps and when it becomes a small-files problem.

In [ ]:
def reset_output_directory(base_dir: Path, overwrite: bool) -> None:
    if base_dir.exists():
        if not overwrite:
            raise FileExistsError(
                f"{base_dir} already exists. Set OVERWRITE=True if you want to regenerate the dataset."
            )
        if "phase2_26L" not in str(base_dir):
            raise ValueError(f"Refusing to delete unexpected path: {base_dir}")
        shutil.rmtree(base_dir)

    UNPARTITIONED_DIR.mkdir(parents=True, exist_ok=True)
    PARTITIONED_DIR.mkdir(parents=True, exist_ok=True)
    DIM_DIR.mkdir(parents=True, exist_ok=True)


def write_partitioned_by_date(df: pl.DataFrame, chunk_id: int) -> None:
    for key, part in df.partition_by("event_date", as_dict=True, include_key=True).items():
        date_value = key[0] if isinstance(key, tuple) else key
        part_dir = PARTITIONED_DIR / f"event_date={date_value}"
        part_dir.mkdir(parents=True, exist_ok=True)
        part.drop("event_date").write_parquet(part_dir / f"part-{chunk_id:05d}.parquet", compression=COMPRESSION)


def parquet_file_stats(base_dir: Path) -> dict[str, int | float]:
    files = list(base_dir.rglob("*.parquet"))
    total_bytes = sum(path.stat().st_size for path in files)
    return {
        "files": len(files),
        "bytes": total_bytes,
        "mib": round(total_bytes / 2**20, 2),
        "avg_file_mib": round(total_bytes / max(1, len(files)) / 2**20, 2),
    }


def write_dataset() -> dict:
    reset_output_directory(BASE_DIR, OVERWRITE)

    users_df.write_parquet(DIM_DIR / "users.parquet", compression=COMPRESSION)
    campaigns_df.write_parquet(DIM_DIR / "campaigns.parquet", compression=COMPRESSION)

    rng = np.random.default_rng(SEED)
    started_at = time.perf_counter()
    rows_written = 0

    for chunk_id, start in enumerate(range(0, NUM_ROWS, CHUNK_SIZE)):
        n = min(CHUNK_SIZE, NUM_ROWS - start)
        df = generate_event_chunk(start + 1, n, rng)

        df.write_parquet(UNPARTITIONED_DIR / f"part-{chunk_id:05d}.parquet", compression=COMPRESSION)
        write_partitioned_by_date(df, chunk_id)

        rows_written += n
        print(f"chunk={chunk_id:03d} rows={n:,} total={rows_written:,}")

    elapsed = round(time.perf_counter() - started_at, 2)

    manifest = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "scale": SCALE,
        "seed": SEED,
        "rows": rows_written,
        "chunk_size": CHUNK_SIZE,
        "compression": COMPRESSION,
        "date_range": {"start": str(START_DATE), "end": str(END_DATE)},
        "environment": {
            "python": sys.version.split()[0],
            "platform": platform.platform(),
            "polars": pl.__version__,
            "numpy": np.__version__,
            "pyarrow": pa.__version__,
            "cpu_logical_cores": psutil.cpu_count(logical=True),
            "ram_gib": round(psutil.virtual_memory().total / 2**30, 2),
        },
        "paths": {
            "base_dir": str(BASE_DIR),
            "events_unpartitioned": str(UNPARTITIONED_DIR),
            "events_partitioned_by_date": str(PARTITIONED_DIR),
            "users": str(DIM_DIR / "users.parquet"),
            "campaigns": str(DIM_DIR / "campaigns.parquet"),
        },
        "layout_stats": {
            "events_unpartitioned": parquet_file_stats(UNPARTITIONED_DIR),
            "events_partitioned_by_date": parquet_file_stats(PARTITIONED_DIR),
            "dimensions": parquet_file_stats(DIM_DIR),
        },
        "generation_seconds": elapsed,
    }

    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return manifest


manifest = write_dataset()
manifest

## Validate the generated dataset

Run this after generation. It checks that both layouts contain the expected number of rows and that required columns are present.

In [ ]:
required_columns = {
    "post_id", "user_id", "event_ts", "ingest_ts", "event_date", "category",
    "country", "device", "language", "campaign_id", "views", "likes", "shares",
    "latency_ms", "upload_error", "content_length", "tags", "schema_version",
    "experiment_group", "ingest_lag_seconds",
}

unpartitioned_scan = pl.scan_parquet(str(UNPARTITIONED_DIR / "*.parquet"))
partitioned_scan = pl.scan_parquet(str(PARTITIONED_DIR / "event_date=*" / "*.parquet"), hive_partitioning=True)

schema_columns = set(unpartitioned_scan.collect_schema().names())
missing = required_columns - schema_columns
assert not missing, f"Missing columns: {sorted(missing)}"

unpartitioned_rows = unpartitioned_scan.select(pl.len()).collect().item()
partitioned_rows = partitioned_scan.select(pl.len()).collect().item()

assert unpartitioned_rows == NUM_ROWS, (unpartitioned_rows, NUM_ROWS)
assert partitioned_rows == NUM_ROWS, (partitioned_rows, NUM_ROWS)

summary = unpartitioned_scan.select(
    pl.len().alias("rows"),
    pl.col("user_id").n_unique().alias("unique_users"),
    pl.col("campaign_id").null_count().alias("campaign_nulls"),
    pl.col("country").null_count().alias("country_nulls"),
    (pl.col("ingest_lag_seconds") > 6 * 3600).sum().alias("late_events_6h_plus"),
    (pl.col("schema_version") == 2).sum().alias("schema_v2_rows"),
).collect()

summary

## Inspect file layout

Use this output in your report. Do not assume that more files are always better. Partitioning can reduce bytes scanned for selective date filters, but too many small files increase scheduling and metadata overhead.

In [ ]:
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
pl.DataFrame(
    [
        {"layout": name, **stats}
        for name, stats in manifest["layout_stats"].items()
    ]
)

## Layout smoke test

This is not the full benchmark yet. The goal is only to verify that both layouts are readable and to prepare intuition for later query-plan analysis.

In [ ]:
DATE_FILTER_START = date(2025, 2, 1)
DATE_FILTER_END = date(2025, 2, 8)

layout_checks = {
    "unpartitioned": unpartitioned_scan,
    "partitioned_by_date": partitioned_scan,
}

results = []
for layout_name, scan in layout_checks.items():
    started = time.perf_counter()
    out = (
        scan
        .filter((pl.col("event_date") >= DATE_FILTER_START) & (pl.col("event_date") < DATE_FILTER_END))
        .group_by("category")
        .agg(
            pl.len().alias("events"),
            pl.col("views").sum().alias("views"),
            pl.col("likes").sum().alias("likes"),
        )
        .sort("events", descending=True)
        .collect()
    )
    results.append({"layout": layout_name, "seconds": round(time.perf_counter() - started, 4), "rows": out.height})

pl.DataFrame(results)

## Questions for the report

Answer briefly:

1. Which layout has more files? Why?
2. For a query filtering one week of data, which layout do you expect to read fewer bytes?
3. For a query scanning all dates and grouping by `category`, which layout may be slower and why?
4. How can user skew affect group-by, join, or shuffle-heavy queries?
5. Why is recording library versions important before comparing this semester's Polars results with the previous semester?

## Next tasks

Later Phase 2 tasks should reuse this dataset for:

- Pandas, Polars, DuckDB, and PySpark local benchmarks,
- old-style vs modern Polars API comparison,
- eager vs lazy vs streaming Polars execution,
- Spark local vs Dataproc cluster scaling,
- query-plan and cost analysis.